# validate_stage3
CPU coordinator notebook. No GPU, Torch or Hugging Face token required. Reuse the accepted Stage 2 suite; this notebook does not extract features or run worker fits. Setup fixes one clean code commit. Run cells in order.


In [ ]:
import os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage3')
CODE_REF = 'phase1'  # Resolved once below; use the published foundation SHA to pin explicitly.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Drazr/Research.git',str(REPO)],check=True)
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git','fetch','origin',CODE_REF],cwd=REPO,check=True)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=REPO,text=True).strip()
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-stage2-cpu.txt')],check=True)
print('Fixed code commit:',CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
TASK = 'validate'
MANIFEST = PHASE1_ROOT / 'REPLACE_STAGE3_PLAN/_COMPLETE.json'
INVENTORY = PHASE1_ROOT / 'REPLACE_INTEGRATED_WORKER_INVENTORY.json'


In [ ]:
import json
if subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()!=CODE_COMMIT or subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Checkout changed after setup.')
command=[sys.executable,'-u','-m','src.stage3.operations',TASK,'--phase1-root',str(PHASE1_ROOT),'--manifest',str(MANIFEST),'--storage-scope','persistent-drive']
if TASK == 'validate':
    if not INVENTORY.is_file(): raise RuntimeError('Set INVENTORY to the explicit integrated worker inventory JSON.')
    command += ['--inventory',str(INVENTORY)]
process=subprocess.Popen(command,cwd=REPO,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:
    print(line,end='',flush=True)
if process.wait()!=0:
    raise RuntimeError('Stage 3 task blocked/failed; inspect the streamed report before continuing.')
